<a href="https://colab.research.google.com/github/empirehq/0xlabs-base-agent-starter-kit/blob/main/NEMOV1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1: A100 Infrastructure Setup
!pip install --upgrade -qqq uv
!uv pip install -qqq "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!uv pip install -qqq "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
# Mamba build (This part will take a few minutes to compile kernels)
!pip install --no-build-isolation mamba-ssm "causal-conv1d>=1.4.0"

import torch
# Safety Check: Confirming A100 and BF16 support
major_version, minor_version = torch.cuda.get_device_capability()
if major_version >= 8:
    print("✅ Ampere GPU Detected (A100/H100). BF16 training is ACTIVE.")
else:
    print("❌ Non-Ampere GPU. Using FP16.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 9.0 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 17.0 MB/s eta 0:00:00
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl size=533592144 sha256=d66a3c4c94a693e02d341cace7a6af0b72177b6afa655a25e3a6505130a68cbf
  Stored in directory: /root/.cache/pip/wheels/28/83/54/d45107838fec575b93f5d723f56351cee19a1b13bcd4ec9f3f
  Created wheel for causal-conv1d: filename=causal_conv1d-1.6.1-cp312-cp312-linux_x86_64.whl size=254017668 sha256=fd2f2d0ccb8208cc3af7c99eb1af7e457ac7b64edaefae8e6c2f3026720e53d8
  Stored in directory: /root/.cache/pip/wheels/98/4a/75/b24971cff4599825b16b612f08fbd2e60a2c336a56e081a3c8
Successfully built mamba-ssm causal-conv1d
✅ Ampere GPU Detected (A100/H100). BF16 training is ACTI

In [5]:
import os, json, time, torch
from google.colab import drive
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

# --- 1. Mount Google Drive (The "Kaggle Lesson" Fail-Safe) ---
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/nemo_claw_v1"
os.makedirs(SAVE_PATH, exist_ok=True)

# --- 2. Data Loading (Path Fixed for Colab) ---
training_file = "/content/final_training_data.jsonl"

raw_examples = []
with open(training_file, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            obj = json.loads(line)
            if 'messages' in obj:
                user_msg = next(m['content'] for m in obj['messages'] if m['role'] == 'user')
                asst_msg = next(m['content'] for m in obj['messages'] if m['role'] == 'assistant')
                raw_examples.append({'text': f"<s>[INST] {user_msg.strip()} [/INST] {asst_msg.strip()}</s>"})
            elif 'prompt' in obj:
                raw_examples.append({'text': f"<s>[INST] {obj['prompt'].strip()} [/INST] {obj['completion'].strip()}</s>"})
        except json.JSONDecodeError:
            continue

print(f"Loaded {len(raw_examples)} examples")

dataset = Dataset.from_list(raw_examples)
split = dataset.train_test_split(test_size=0.05, seed=42)

# --- 3. Model Loading (A100-80GB Specs) ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/NVIDIA-Nemotron-3-Nano-4B',
    max_seq_length=8192,
    dtype=torch.bfloat16,        # Native A100 stability
    load_in_4bit=True,
    trust_remote_code=True,      # FIXED: Required for Nemotron-3
)

model = FastLanguageModel.get_peft_model(
    model,
    r=64,                        # Reasoning capacity for 49k examples
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=64,
    lora_dropout=0,              # FIXED: Our proven setting
    bias="none",                 # FIXED: Our proven setting
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

# --- 4. Training Arguments (Claude's Surgical Fixes Applied) ---
training_args = SFTConfig(
    output_dir=f"{SAVE_PATH}/checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=32, # If OOM, drop to 16
    gradient_accumulation_steps=1,
    warmup_steps=50,                # FIXED: Prevents early loss spikes at batch 32
    learning_rate=2e-4,
    bf16=True,                      # Native A100 BF16
    logging_steps=5,
    eval_strategy='steps',          # FIXED: Evals will actually run now
    eval_steps=100,
    save_strategy='steps',          # FIXED: Checkpoints will actually save
    save_steps=250,
    save_total_limit=2,
    optim='adamw_8bit',
    max_seq_length=4096,
    dataset_text_field='text',
    report_to='none',               # FIXED: Clean logs
    seed=42,                        # FIXED: Reproducibility
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    tokenizer=tokenizer,
)

print(f"Training {len(split['train'])} examples on A100-80GB...")
t_start = time.time() # FIXED: Timing printout start
trainer.train()
print(f"Training complete! {(time.time()-t_start)/60:.1f} minutes") # FIXED: Timing printout end

# --- 5. Save GGUF directly to Google Drive ---
print("Exporting GGUF to Google Drive...")
model.save_pretrained_gguf(
    f"{SAVE_PATH}/Nemo-Claw-v1-GGUF",
    tokenizer,
    quantization_method='q4_k_m',
)

print(f"DONE! Your model is safe in your Google Drive: {SAVE_PATH}/Nemo-Claw-v1-GGUF")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 49238 examples
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.17: Fast Nemotron_H patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Nemotron_H does not support SDPA - switching to fast eager.


Loading weights:   0%|          | 0/263 [00:00<?, ?it/s]

Unsloth: Making `model.base_model.model.backbone` require gradients


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/46776 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/2462 [00:00<?, ? examples/s]

...

Unsloth: Copying 1 files from cache to `/content/drive/MyDrive/nemo_claw_v1/Nemo-Claw-v1-GGUF`: 100%|██████████| 1/1 [00:21<00:00, 21.62s/it]


Successfully copied all 1 files from cache to `/content/drive/MyDrive/nemo_claw_v1/Nemo-Claw-v1-GGUF`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:34<00:00, 34.51s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/nemo_claw_v1/Nemo-Claw-v1-GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/nemo_claw_v1/Nemo-Claw-v1-GGUF_gguf/NVIDIA-Nemotron-3-Nano-4B.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/nemo_claw_v1/Nemo-Claw-v1-GGUF_gguf/NVIDIA-Nemotron-3-Nano-4B.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/NVIDIA-Nemotron-3-Nano-4B'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/nemo_claw_v1/Nemo-Claw-v1-GGUF_gguf/NVIDIA-Nemotron-3-Nano-4B.Q4_K_M.gguf -p "why is the sky blue?"
DONE! Your model is safe in your Google Drive: /content/drive/MyDrive/nemo_claw_v1/Nemo-Claw-v1-GGUF
